# Plot phenotype cross-products -- per trait, coloured by covariate-set model

Reads the merged `<phenotype>__<transform>__<covariate_set>_merged.full.tsv` files that
`grm_shard_processing.ipynb` / `grm_shard_accumulate_parallel.ipynb` already produced and
uploaded to `CROSSPRODUCTS_BUCKET_GS`. For each phenotype (one plot per `transform`), draws
mean cross-product vs. GRM relatedness bin, one coloured line per covariate-set model (`base`,
`base_pcs`, `base_pcs_zip3`, `base_pcs_zip3_ses` -- whatever's actually present), in `ggplot2`.

Every plot is both shown inline and uploaded back up to
`{BUCKET_DIR_GS}/data/04_process_shards/crossproducts/plots/` -- doesn't touch the `.tsv`
inputs sitting alongside it in the same bucket directory.

## Setup

In [ ]:
required_pkgs <- c("dplyr", "readr", "tidyr", "stringr", "ggplot2")
missing_pkgs <- required_pkgs[!sapply(required_pkgs, requireNamespace, quietly = TRUE)]
if (length(missing_pkgs) > 0) install.packages(missing_pkgs)

library(dplyr)
library(readr)
library(tidyr)
library(stringr)
library(ggplot2)

theme_set(theme_bw(base_size = 12))
options(repr.plot.width = 8, repr.plot.height = 6)

## Paths

`CROSSPRODUCTS_DIR` is the local (FUSE-mounted) view of the same bucket directory both
processing notebooks write `*_merged.full.tsv` into -- read locally (no `gsutil`/`gcloud`
download needed), only the *plots* get pushed back up via `gcloud storage cp` at the end.
`SLOPE_FIT_RANGE` mirrors `grm_shard_processing.ipynb`'s convention -- not used for a slope
line here (kept simple/ggplot-native via `geom_smooth`), just documented for reference.

In [ ]:
WORKSPACE_BUCKET <- path.expand("~/workspace/Data from All of Us Controlled Tier /shared-env-pilot")
WORKSPACE_BUCKET_GS <- "gs://shared-env-pilot-wb-fulgent-almond-9474"

CROSSPRODUCTS_DIR <- file.path(WORKSPACE_BUCKET, "data", "04_process_shards", "crossproducts")
CROSSPRODUCTS_BUCKET_GS <- paste0(WORKSPACE_BUCKET_GS, "/data/04_process_shards/crossproducts")
PLOTS_BUCKET_GS <- paste0(CROSSPRODUCTS_BUCKET_GS, "/plots")

LOCAL_PLOTS_DIR <- path.expand("~/grm_pheno_cov/plots_ggplot")
dir.create(LOCAL_PLOTS_DIR, recursive = TRUE, showWarnings = FALSE)

stopifnot(dir.exists(CROSSPRODUCTS_DIR))
sprintf("Reading merged crossproducts from: %s", CROSSPRODUCTS_DIR)

## Discover + load merged files

Filenames are `<phenotype>__<transform>__<covariate_set>_merged.full.tsv` (the `tag` +
`_merged.full.tsv` suffix both processing notebooks write). Parsed the same
`rsplit("__", 2)`-equivalent way `grm_shard_processing.ipynb` parses `.pheno` filenames, so a
new covariate-set or transform name doesn't need any code change here.

In [ ]:
merged_files <- list.files(CROSSPRODUCTS_DIR, pattern = "_merged\\.full\\.tsv$", full.names = TRUE)
stopifnot(length(merged_files) > 0)

parse_tag <- function(path) {
  stem <- str_remove(basename(path), "_merged\\.full\\.tsv$")
  parts <- str_split(stem, "__", n = 3)[[1]]
  if (length(parts) != 3) return(NULL)
  list(phenotype_name = parts[1], transform = parts[2], covariate_set = parts[3])
}

all_data <- bind_rows(lapply(merged_files, function(path) {
  tag <- parse_tag(path)
  if (is.null(tag)) {
    message(sprintf("Skipping unparseable filename: %s", basename(path)))
    return(NULL)
  }
  read_tsv(path, show_col_types = FALSE) %>%
    mutate(phenotype_name = tag$phenotype_name, transform = tag$transform,
           covariate_set = tag$covariate_set, .before = 1)
}))

phenotypes <- sort(unique(all_data$phenotype_name))
covariate_sets <- sort(unique(all_data$covariate_set))
sprintf("%d phenotypes, %d covariate-set models, %d merged files loaded",
        length(phenotypes), length(covariate_sets), length(merged_files))

## Plot + save + upload, per phenotype

One plot per `(phenotype, transform)`, `covariate_set` (the covariate-set "model") as colour -- error bars
from `jk_se`, only bins with `full_n > 0`. Saved locally then uploaded to `PLOTS_BUCKET_GS`
with `gcloud storage cp` right after each plot (not batched at the end), same
upload-as-you-go convention as the accumulate step itself.

In [ ]:
covset_colors <- setNames(
  scales::viridis_pal()(length(covariate_sets)),
  covariate_sets
)

plot_phenotype_transform <- function(phenotype_name, transform, d) {
  d <- d %>% filter(full_n > 0)
  d$covariate_set <- factor(d$covariate_set, levels = covariate_sets)

  p <- ggplot(d, aes(x = bin_midpoint, y = full_mean, color = covariate_set)) +
    geom_hline(yintercept = 0, color = "grey70", linewidth = 0.4) +
    geom_vline(xintercept = 0, color = "grey70", linewidth = 0.4) +
    geom_errorbar(aes(ymin = full_mean - jk_se, ymax = full_mean + jk_se), width = 0, alpha = 0.6) +
    geom_point(size = 1.3) +
    geom_line(linewidth = 0.4, alpha = 0.7) +
    scale_color_manual(values = covset_colors) +
    labs(
      x = "GRM relatedness (bin midpoint)",
      y = "Mean phenotype cross-product (covariance)",
      color = "Covariate-set model",
      title = sprintf("%s (%s) -- covariance vs relatedness", phenotype_name, transform)
    ) +
    theme(plot.title = element_text(hjust = 0.5), legend.position = "top")

  out_file <- sprintf("%s__%s.png", phenotype_name, transform)
  local_path <- file.path(LOCAL_PLOTS_DIR, out_file)
  ggsave(local_path, p, width = 8, height = 6, dpi = 150)

  print(p)   # inline, right here -- visible before the next phenotype starts

  system2("gcloud", c("storage", "cp", shQuote(local_path), shQuote(paste0(PLOTS_BUCKET_GS, "/"))))
  local_path
}

saved_paths <- c()
for (i in seq_along(phenotypes)) {
  phenotype_name <- phenotypes[i]
  message(sprintf("=== [%d/%d] %s ===", i, length(phenotypes), phenotype_name))
  pheno_data <- all_data %>% filter(phenotype_name == !!phenotype_name)
  for (transform in sort(unique(pheno_data$transform))) {
    d <- pheno_data %>% filter(transform == !!transform)
    saved_paths <- c(saved_paths, plot_phenotype_transform(phenotype_name, transform, d))
  }
}

sprintf("done -- %d plots saved locally to %s and uploaded to %s",
        length(saved_paths), LOCAL_PLOTS_DIR, PLOTS_BUCKET_GS)

## Copying already-existing plots to the bucket

If you've already got PNGs sitting locally from an earlier run of
`grm_shard_processing.ipynb` (`~/grm_pheno_cov/plots/`) or this notebook
(`~/grm_pheno_cov/plots_ggplot/`), push them up the same way without re-plotting anything --
run in a terminal on the VM, or as a shell cell here:

```bash
gcloud storage cp ~/grm_pheno_cov/plots/*.png \
  gs://shared-env-pilot-wb-fulgent-almond-9474/data/04_process_shards/crossproducts/plots/
gcloud storage cp ~/grm_pheno_cov/plots_ggplot/*.png \
  gs://shared-env-pilot-wb-fulgent-almond-9474/data/04_process_shards/crossproducts/plots/
```

Plain `cp`/`gsutil cp` also both work here (unlike the same-bucket-copy case noted in
`grm_shard_batch_submit.ipynb`'s dsub work -- see repo memory) since this is a local-to-GCS
upload, not a same-bucket server-side copy; `gcloud storage cp` is used above purely for
consistency with the rest of this notebook.